# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## Task type: Scoring / Ranking

This is a **scoring** problem, not classification. I'm not sorting pages into
fixed buckets — I want a continuous score per page that lets me rank all
declining pages from "review first" to "review last." The model's job is to
produce that ordering, not a hard yes/no label.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "Scoring / Ranking"
print(f"Task type: {task_type}")

Task type: Scoring / Ranking


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target / proxy

There's no ground-truth "should be refreshed" label — no one hand-labeled
pages that way. I need a **proxy target**: `is_declining_label`
(trend_direction == "down"), the same one from Week 1.

This is a proxy, not a real outcome label, because a page can decline for
reasons a refresh won't fix (seasonality, a client dropping the page). I'm
only claiming this label means "worth a look," not "refresh will help."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess, pandas as pd

if not os.path.isdir("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/MuhammadAliEjaz1/flyrank-ml-internship-starter"], check=True)
os.chdir("flyrank-ml-internship-starter")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success metric: Precision@K

An editor only reviews the top K pages (say K=50), so what matters is: of the
top 50 pages my score ranks highest, how many are actually declining? Overall
accuracy across all 30,000 pages would reward being right about pages nobody
will ever review. Precision@K directly matches how the output gets used.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k):
    order = pd.Series(scores).sort_values(ascending=False).index[:k]
    return labels[order].mean()

# quick preview using impressions as a placeholder score, just to show the metric works
preview_precision = precision_at_k(df["impressions_90d"], df["is_declining_label"], 50)
print(f"Precision@50 (impressions-only baseline): {preview_precision:.3f}")

Precision@50 (impressions-only baseline): 0.420


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## Unit of analysis

One row = one page's current snapshot — not a client, not a day. This matches
the starter dataset's grain directly.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_of_analysis = df[[
    "trend_direction", "impressions_90d", "avg_position",
    "days_since_last_update", "content_type", "is_declining_label"
]].head(10)
unit_of_analysis

,trend_direction,impressions_90d,avg_position,days_since_last_update,content_type,is_declining_label
0,down,3803,10.6,20,keyword article,1
1,down,15320,20.3,25,keyword article,1
2,down,12581,36.5,20,keyword article,1
3,stable,11751,6.2,22,keyword article,0
4,down,19140,44.0,14,keyword article,1
5,down,3970,8.5,20,keyword article,1
6,down,20,7.0,20,keyword article,1
7,stable,1724,21.2,22,keyword article,0
8,down,32574,46.0,20,keyword article,1
9,down,1240,4.9,104,keyword article,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML beats a fixed rule here

Week 1 showed declining, visible pages have a median of only 22 days since
last update — shorter than expected if staleness alone explained decline. A
hand rule like "stale AND visible" can't capture interactions between
staleness, position, and traffic that don't move together. A learned model
can weigh multiple signals at once instead of me guessing the right
combination and threshold by hand.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
reasons = ["staleness alone doesn't explain decline",
           "position and traffic interact in ways a single rule can't capture",
           "a model can weigh signals jointly, not one threshold at a time"]
for r in reasons:
    print("-", r)

- staleness alone doesn't explain decline
- position and traffic interact in ways a single rule can't capture
- a model can weigh signals jointly, not one threshold at a time


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.